# PARC2026 — M2/M3 Common Model Benchmark Controller

D10で決まった **同じ provisional best dataset** を使い、`π0.5 / SmolVLA / OpenVLA-OFT` を同じ評価契約で比較するためのcontrollerです。

正式比較は2系統です。
1. **Equal Data Exposure** — 同じepisode pool / sampling policy / comparable samples seen
2. **Equal Wall Time** — 同じA100 training-loop wall-time budget

screening budgetは `experiments/plans/model_benchmark_budget_v1.json` に固定します。現行v1は 4,800 samples/model と 1,800 sec/model（training-loopのみ）です。

OpenVLA-OFTは、同じselected episode poolを表す **full RLDS conversion contract または69c validated LeRobot streaming contract** のどちらかが必要です。別datasetを代用して公平比較と見なしません。69bでfull RLDSが56.61 GiBと見積もられた現行経路では69c streaming contractを優先します。

**注意:** このNotebookはまだM2/M3 controllerです。GateがREADYになっても3モデルの実学習runnerは別途接続するまで開始しません。


In [ ]:
# 0/5 Preflight + D10 decision gate
import os, json, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
try:
    tok=os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
except Exception:
    tok=None
if not tok: raise RuntimeError('Colab Secretsに HF_TOKEN を登録してください。')
os.environ['HF_TOKEN']=tok
gpu_name=subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True).strip()
vram=int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],text=True).strip())
print('GPU:',gpu_name,vram,'MiB')
if vram < 38000: raise RuntimeError('M2/M3 training bring-upはA100 40GB以上を使用してください。L4は69/69b/69cと最終inference gate用です。')
DRIVE=Path('/content/drive/MyDrive/parc2026-cache')
decision_path=DRIVE/'pi05-top2-tiebreak-v1/provisional_best_dataset_recipe.json'
assert decision_path.exists(),decision_path
decision=json.loads(decision_path.read_text())
if decision.get('status')!='DECIDED': raise RuntimeError(f'D10未決着: {decision}')
SELECTED_VARIANT=decision['selected_variant']
assert SELECTED_VARIANT=='V2_SQRT_BALANCED_RAW',decision
print('SELECTED DATASET:',SELECTED_VARIANT)
print('=== D10 GATE: PASS ===')


In [ ]:
# 1/5 Clone repo + model registry + frozen budget
import json, subprocess
from pathlib import Path
ROOT=Path('/content/parc2026'); REPO=ROOT/'py_AI'; ROOT.mkdir(parents=True,exist_ok=True)
if not (REPO/'.git').exists(): subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--force','origin/main'],check=True)
DRIVE_OUT=DRIVE/'model-benchmark-v1'; DRIVE_OUT.mkdir(parents=True,exist_ok=True)
registry=json.loads((REPO/'experiments/model_registry_v1.json').read_text())
budget=json.loads((REPO/'experiments/plans/model_benchmark_budget_v1.json').read_text())
assert budget['status']=='FROZEN' and budget['selected_dataset_variant']==SELECTED_VARIANT,budget
EQUAL_DATA_SAMPLE_BUDGET=int(budget['equal_data_exposure']['sample_budget'])
EQUAL_WALL_TIME_SEC=int(budget['equal_wall_time']['train_loop_sec'])
assert EQUAL_DATA_SAMPLE_BUDGET==4800 and EQUAL_WALL_TIME_SEC==1800,budget
manifest={'schema_version':2,'stage':'M2_candidate_bringup','selected_dataset_variant':SELECTED_VARIANT,'dataset_id':'lerobot/libero_plus','dataset_revision':'f3f49f426d75030177b18778374005bc12ccd588','models':registry['models'],'m3_budget':budget,'eval_contract':{'tracks_for_screening':['track1','track2'],'max_steps':300,'seed_set':[20260906,20260907],'same_task_selection':True,'same_n_episodes_per_task':True,'final_l4_24gb_inference_gate':True}}
(DRIVE_OUT/'model_bringup_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('Equal Data Exposure:',EQUAL_DATA_SAMPLE_BUDGET,'samples/model')
print('Equal Wall Time:',EQUAL_WALL_TIME_SEC,'sec/model training-loop')
print('=== MODEL REGISTRY + BUDGET: PASS ===')


In [ ]:
# 2/5 Stage the same LeRobot source and verify the fixed D10 group-aware manifest
import json, subprocess
from pathlib import Path
src=DRIVE/'datasets/lerobot_libero_plus_v3_train'; dst=ROOT/'datasets/public_libero_plus_v3_train'
assert (src/'meta/info.json').exists(),src
if not (dst/'meta/info.json').exists():
    dst.parent.mkdir(parents=True,exist_ok=True); print('=== Drive -> local dataset stage ===',flush=True)
    subprocess.run(['rsync','-a','--info=progress2',str(src)+'/',str(dst)+'/'],check=True)
else:
    print('dataset already staged:',dst)
stats=json.loads((dst/'meta/stats.json').read_text())
for f in ['observation.state','action']: assert 'q01' in stats[f] and 'q99' in stats[f]
print('source q01/q99 metadata: PASS')

SELECTED_MANIFEST=DRIVE/'pi05-ablation-group-aware-v2/dataset_ablation_manifests_v2_group_aware/V2_SQRT_BALANCED_RAW.json'
assert SELECTED_MANIFEST.exists(),SELECTED_MANIFEST
selected=json.loads(SELECTED_MANIFEST.read_text())
EXPECTED_SELECTED_SHA='73ed0d3b0c5e73c745c0aa2e81517ce1fa40240c75f9eb040d65b6876ba08239'
assert selected['schema_version']==2 and selected['group_aware'] is True and selected['variant']==SELECTED_VARIANT
assert selected['dataset_id']=='lerobot/libero_plus'
assert selected['dataset_revision']=='f3f49f426d75030177b18778374005bc12ccd588'
assert selected['episode_ids_sha256']==EXPECTED_SELECTED_SHA
EPISODE_COUNT=selected['summary']['episode_count']; FRAME_COUNT=selected['summary']['frame_count']
assert EPISODE_COUNT==10758 and FRAME_COUNT==1620614
print('fixed manifest:',SELECTED_MANIFEST)
print('episodes:',EPISODE_COUNT,'frames:',FRAME_COUNT,'sha:',EXPECTED_SELECTED_SHA)
print('=== COMMON DATASET GATE: PASS ===')


In [ ]:
# 3/5 Model source bring-up + exact selected-subset bridge gate
import json, subprocess
from pathlib import Path
VENDOR=ROOT/'vendor'; VENDOR.mkdir(parents=True,exist_ok=True)
pins={
    'smolvla':{
        'url':'https://github.com/huggingface/lerobot.git',
        'sha':'3f2c29ef7e44b1ddccbcda3b6a63939e53639e9e',
        'path':VENDOR/'lerobot-smolvla',
        'required':'src/lerobot'
    },
    'openvla_oft':{
        'url':'https://github.com/small-zeng/openvla-oft.git',
        'sha':'e4287e94541f459edc4feabc4e181f537cd569a8',
        'path':VENDOR/'openvla-oft',
        'required':'vla-scripts/finetune.py'
    }
}
for name,cfg in pins.items():
    path=cfg['path']
    if not (path/'.git').exists():
        subprocess.run(['git','init','-q',str(path)],check=True)
        subprocess.run(['git','-C',str(path),'remote','add','origin',cfg['url']],check=True)
    subprocess.run(['git','-C',str(path),'fetch','-q','--depth','1','origin',cfg['sha']],check=True)
    subprocess.run(['git','-C',str(path),'checkout','-q','--force','FETCH_HEAD'],check=True)
    got=subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()
    assert got==cfg['sha'],(name,got)
    assert (path/cfg['required']).exists()
    print(name,'@',got)

full_contract_path=DRIVE/'openvla-rlds-selected-v1/conversion_contract.json'
stream_contract_path=DRIVE/'openvla-streaming-selected-v1/streaming_bridge_contract.json'
bridge_contract=None
bridge_type=None

if full_contract_path.is_file():
    candidate=json.loads(full_contract_path.read_text())
    assert candidate.get('status')=='PASS',candidate
    assert candidate.get('selected_dataset_variant')==SELECTED_VARIANT
    assert candidate.get('source_episode_ids_sha256')==selected['episode_ids_sha256']
    assert candidate.get('converted_episode_count')==EPISODE_COUNT
    bridge_contract=candidate
    bridge_type='full_rlds'
elif stream_contract_path.is_file():
    candidate=json.loads(stream_contract_path.read_text())
    assert candidate.get('status')=='PASS',candidate
    assert candidate.get('bridge_type')=='lerobot_streaming'
    assert candidate.get('selected_dataset_variant')==SELECTED_VARIANT
    assert candidate.get('source_dataset_id')=='lerobot/libero_plus'
    assert candidate.get('source_dataset_revision')=='f3f49f426d75030177b18778374005bc12ccd588'
    assert candidate.get('source_episode_ids_sha256')==selected['episode_ids_sha256']
    assert candidate.get('source_episode_count')==EPISODE_COUNT
    assert candidate.get('source_frame_count')==FRAME_COUNT
    assert candidate.get('openvla_oft_repo')=='small-zeng/openvla-oft'
    assert candidate.get('openvla_oft_revision')==pins['openvla_oft']['sha']
    oc=candidate.get('openvla_contract',{})
    assert oc.get('action_dim')==7 and oc.get('proprio_dim')==8 and oc.get('action_chunk')==8
    assert oc.get('normalization_type')=='bounds_q99'
    assert oc.get('absolute_action_mask')==[False]*6+[True]
    assert oc.get('action_normalization_mask')==[True]*6+[False]
    ds=candidate.get('dataset_statistics',{})
    assert ds.get('num_trajectories')==EPISODE_COUNT and ds.get('num_transitions')==FRAME_COUNT
    assert candidate.get('storage_policy',{}).get('full_rlds_materialized') is False
    bridge_contract=candidate
    bridge_type='lerobot_streaming'

openvla_ready=bridge_contract is not None
status={
    'pi05':{'code':'PASS','dataset_format':'lerobot'},
    'smolvla':{'code':'PASS','dataset_format':'lerobot'},
    'openvla_oft':{
        'code':'PASS',
        'dataset_format':'rlds_or_validated_streaming_bridge',
        'selected_subset_bridge_contract':'PASS' if openvla_ready else 'MISSING',
        'bridge_type':bridge_type,
        'contract_path':str(full_contract_path if bridge_type=='full_rlds' else stream_contract_path) if openvla_ready else None,
    }
}
(DRIVE_OUT/'bringup_status.json').write_text(json.dumps(status,indent=2)+'\n')
print(json.dumps(status,indent=2))
if not openvla_ready:
    print('\nM2 BLOCKED: exact selected episode-pool OpenVLA bridge contract is missing.')
    print('Run 69c streaming validation (or provide an exact full RLDS conversion contract) before M3.')
else:
    print('OpenVLA selected-subset bridge contract: PASS via',bridge_type)


In [ ]:
# 4/5 Freeze comparison protocol from repository contract
import json
status=json.loads((DRIVE_OUT/'bringup_status.json').read_text())
protocol={
    'schema_version':2,
    'selected_dataset_variant':SELECTED_VARIANT,
    'selected_episode_count':EPISODE_COUNT,
    'selected_frame_count':FRAME_COUNT,
    'selected_episode_ids_sha256':selected.get('episode_ids_sha256'),
    'openvla_bridge_type':status['openvla_oft'].get('bridge_type'),
    'equal_data_exposure':{
        'sample_budget':EQUAL_DATA_SAMPLE_BUDGET,
        'same_episode_pool':True,
        'same_sampling_policy':True
    },
    'equal_wall_time':{
        'a100_train_loop_sec':EQUAL_WALL_TIME_SEC,
        'exclude_setup_download_conversion_eval':True
    },
    'required_metrics':budget['evaluation']['required_metrics'],
    'promotion':budget['promotion'],
    'status':'BLOCKED'
}
reasons=[]
if status['openvla_oft']['selected_subset_bridge_contract']!='PASS':
    reasons.append('openvla_selected_subset_bridge_missing')
if not reasons:
    protocol['status']='READY_FOR_RUNNER'
protocol['blocked_reasons']=reasons
(DRIVE_OUT/'comparison_protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print(json.dumps(protocol,indent=2))
if reasons:
    print('\nM3はまだ開始しません。69c/bridge gateを解消してください。')
else:
    print('=== M3 COMPARISON CONTRACT: READY_FOR_RUNNER ===')
    print('Controller gate passed. Do not infer that training ran; connect the 3-model job runner next.')
